## Phase 0 — Already Done

- data collection
  - NERRS (1995–2025, most regions minus hudson river and 2x great lakes, all stations)
    - originally 15min resolution
    - mostly intact water conditions
    - very sparse nutrient data
      - nutrient data extended ("as-of" forward-fill, 7-day cap)
      - this just in case a record at 945am was skipped over by picking the 1000am record instead to collate at 1hr resolution
    - entirely blank meteorlogical data?
  - ERA5 atmospherics
    - get that atmospheric data back
    - 1hr resolution
    - 0.25 degree resolution
      - not a perfect match to each station in nerrs
      - median centroid was calculated for all stations in a region
      - though most times that centroid ended up nearby but over land (so switched to skin temp instead of water surface temp)
- data cleaning and collation
- split into two datasets
  - full (with nutrients, less complete timeline)
  - and core (without nutrients, more complete)
  - script can be rerun for 1hr resolution
  - but 12hr resolution is included in this repo (zipped)
- various metrics analysis, exploration

In [ ]:
# first... dependencies
import numpy as np                  # for arrays and math
import pandas as pd                 # for dataframes and csv I/O
import matplotlib.pyplot as plt     # basic visualizations
import seaborn as sns               # for quick readable charts

# keep charts easy to read
sns.set_theme( style = 'whitegrid' )

# the files (thee have been collated and cleaned already)
res = 1 # hours - alternatives 4 and 12

# https://www.youtube.com/watch?v=_W7K79FjI58
# mandatory listening while working on this project

In [ ]:
water = pd.read_csv( f'../data/{res}hr/t4d.{res}hr.water.history.csv' )

## Phase 1 — Characterization & Classification
Goal: understand what we have before modeling anything

- 1.1 compute per-station summary statistics over a defined baseline period (suggest 1995–2005)
  - mean annual water temp, seasonal amplitude, mean salinity, mean DO
- 1.2 cluster stations in (mean temp × seasonal amplitude) space 
  - k-means, try k=3 or 4, use elbow/silhouette to let the data suggest the right number of groups
  - example in nutrient analysis work
- 1.3 enrich clusters with any available station metadata (estuary type, distance from mouth, watershed area)
  - confirm clusters are physically meaningful, not just statistical artifacts
- 1.4 assign each station a baseline regime label
  - see if kmeans self-identify... 
  - otherwise probably get a rolling means (temp) per station for 1995-2000 to classify
- 1.5 visualize stations on a map colored by regime
  - sanity check that PR/HI are warm, alaska/maine are cold, etc.
- 1.6 document baseline period statistics per regime as a reference table
  - this will be needed for the paper/poster later, too

In [ ]:
# 1.0 - a description
water.describe().round(3).T

In [ ]:
# lets make this more readable
water = water.rename( columns = {
    'w_temp_c': 'water_temp',
    'w_sal_psu': 'salinity',
    'w_do_mg_l': 'oxygen',
    'w_do_pct': 'oxy_saturation',
    'depth_m': 'depth',
    'w_ph': 'ph',
    'm_wind_ms': 'wind_speed',
    'm_ssrd_kwh_m2': 'solar_radiation',
    'm_precip_mmh': 'precipitation',
    'm_temo_c': 'air_temp'
} )

In [ ]:
# 1.1 - means per station per metric
# the point here is to establish 'the character' of each station in the data
# and, by that.. its water properties.  What is the location's water LIKE, normally.
base = water.copy()
base[ 'datetime' ] = pd.to_datetime( base[ 'datetime' ], errors = 'coerce' )
# just nab the first 5 years for the next step
base = base.loc[ 
    ( base[ 'datetime' ] >= '1995-01-01' ) &
    ( base[ 'datetime' ] < '2001-01-01' ),
    [ 'region', 'station', 'datetime', 
      'water_temp', 'salinity', 'oxygen',
      'oxy_saturation', 'ph', 'depth' ],
].dropna( subset = [ 'datetime' ] )

In [ ]:
# okay, so based on this selection, let's build out some means
base[ 'year' ] = base[ 'datetime' ].dt.year
base[ 'month' ] = base[ 'datetime' ].dt.month
# get mean of each year per station, then the mean of those five years per station
# yearly means first ( equal-year weighting )
annual = (
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        water_temp_ymean = ( 'water_temp', 'mean' ),
        salinity_ymean = ( 'salinity', 'mean' ),
        oxygen_ymean = ( 'oxygen', 'mean' ),
        saturation_ymean = ( 'oxy_saturation', 'mean' ),
        ph_ymean = ( 'ph', 'mean' ),
        depth_ymean = ( 'depth', 'mean' ),
    )
)

annual_means = (
    annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg(
        mean_annual_water_temp = ( 'water_temp_ymean', 'mean' ),
        mean_annual_salinity = ( 'salinity_ymean', 'mean' ),
        mean_annual_oxygen = ( 'oxygen_ymean', 'mean' ),
        mean_annual_saturation = ( 'saturation_ymean', 'mean' ),
        mean_annual_ph = ( 'ph_ymean', 'mean' ),
        mean_annual_depth = ( 'depth_ymean', 'mean' ),
        n_years_water_temp = ( 'water_temp_ymean', 'count' ),
        n_years_salinity = ( 'salinity_ymean', 'count' ),
        n_years_oxygen = ( 'oxygen_ymean', 'count' ),
        n_years_saturation = ( 'saturation_ymean', 'count' ),
        n_years_ph = ( 'ph_ymean', 'count' ),
        n_years_depth = ( 'depth_ymean', 'count' ),
    )
)

# seasonal amplitudes from monthly climatology
monthly = (
    base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg(
        water_temp_mmean = ( 'water_temp', 'mean' ),
        salinity_mmean = ( 'salinity', 'mean' ),
        oxygen_mmean = ( 'oxygen', 'mean' ),
        saturation_mmean = ( 'oxy_saturation', 'mean' ),
        ph_mmean = ( 'ph', 'mean' ),
        depth_mmean = ( 'depth', 'mean' ),
    )
)

# building to a new feature, the 'swing' the mean swing of seasonal properties
seasonal_amp = (
    monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg(
        seasonal_amp_water_temp = ( 'water_temp_mmean', lambda s: s.max() - s.min() ),
        seasonal_amp_salinity = ( 'salinity_mmean', lambda s: s.max() - s.min() ),
        seasonal_amp_oxygen = ( 'oxygen_mmean', lambda s: s.max() - s.min() ),
        seasonal_amp_saturation = ( 'saturation_mmean', lambda s: s.max() - s.min() ),
        seasonal_amp_ph = ( 'ph_mmean', lambda s: s.max() - s.min() ),
        seasonal_amp_depth = ( 'depth_mmean', lambda s: s.max() - s.min() ),
    )
)

# depth summary ( useful if sensor environment differs by station )
# let's see if the recorders are variable depth per station, or predictable?
depth_stats = (
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg(
        median_depth = ( 'depth', 'median' ),
        p10_depth = ( 'depth', lambda s: s.quantile( 0.10 ) ),
        p90_depth = ( 'depth', lambda s: s.quantile( 0.90 ) ),
    )
)
depth_stats[ 'iqr_depth' ] = depth_stats[ 'p90_depth' ] - depth_stats[ 'p10_depth' ]

coverage = (
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg(
        n_obs_total = ( 'datetime', 'size' ),
        n_years_present = ( 'year', 'nunique' ),
    )
)

station_baseline = (
    annual_means
    .merge( seasonal_amp, on = [ 'region', 'station' ], how = 'left' )
    .merge( depth_stats, on = [ 'region', 'station' ], how = 'left' )
    .merge( coverage, on = [ 'region', 'station' ], how = 'left' )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

In [ ]:
# ah, but wait ... not ever station started operating in the 1995-2000 window...
year_obs = (
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( n_obs = ( 'datetime', 'size' ) )
)

year_obs[ 'expected_obs' ] = np.where(
    year_obs[ 'year' ] % 4 == 0,
    366 * 24,
    365 * 24,
)

year_obs[ 'coverage_frac' ] = year_obs[ 'n_obs' ] / year_obs[ 'expected_obs' ]
year_obs[ 'year_is_valid' ] = year_obs[ 'coverage_frac' ] >= 0.70

# station eligibility (do they enough years?)
station_valid = (
    year_obs
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg(
        n_valid_years = ( 'year_is_valid', 'sum' ),
        mean_year_coverage = ( 'coverage_frac', 'mean' ),
    )
)

eligible_stations = station_valid.loc[
    station_valid[ 'n_valid_years' ] >= 4,
    [ 'region', 'station' ],
]

base_eligible = base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )

In [ ]:
eligible_stations

In [ ]:
# tall small-multiples: one subplot per region, one line per station

plot_df = water.copy()

plot_df[ 'datetime' ] = pd.to_datetime( plot_df[ 'datetime' ], errors = 'coerce' )
plot_df = plot_df.dropna( subset = [ 'datetime' ] )

temp_col = 'water_temp' if 'water_temp' in plot_df.columns else 'w_temp_c'

plot_df[ 'year' ] = plot_df[ 'datetime' ].dt.year

annual_station = (
    plot_df
    .groupby( [ 'region', 'station', 'year' ], as_index = False )[ temp_col ]
    .mean()
    .rename( columns = { temp_col: 'mean_annual_temp' } )
)

regions = sorted( annual_station[ 'region' ].dropna().unique() )
n_regions = len( regions )

fig, axes = plt.subplots(
    n_regions,
    1,
    figsize = ( 16, max( 3 * n_regions, 10 ) ),
    sharex = True,
    constrained_layout = True,
)

if n_regions == 1:
    axes = [ axes ]

for ax, region in zip( axes, regions ):
    sub = annual_station.loc[ annual_station[ 'region' ] == region ].sort_values( [ 'station', 'year' ] )

    for station, g in sub.groupby( 'station' ):
        ax.plot(
            g[ 'year' ],
            g[ 'mean_annual_temp' ],
            linewidth = 1.4,
            alpha = 0.85,
            label = station,
        )

    ax.set_title( f'{region}: Mean Annual Water Temp by Station' )
    ax.set_ylabel( 'Temp ( C )' )

    # keep legends readable
    n_stations = sub[ 'station' ].nunique()
    if n_stations <= 12:
        ax.legend( ncol = 3, fontsize = 8, frameon = False, loc = 'upper left' )
    else:
        ax.text(
            0.99,
            0.95,
            f'{n_stations} stations',
            transform = ax.transAxes,
            ha = 'right',
            va = 'top',
            fontsize = 8,
        )

axes[ -1 ].set_xlabel( 'Year' )
plt.show()


## Phase 2 — Temporal Diagnostics
Goal: characterize lag structure and trends before building predictive features

- 2.1 run STL decomposition on water temperature per station
  - annual + diurnal cycles 
  - extract trend components
  - see trends analysis for example
- 2.2 compute cross-correlations between air temp and water temp across a range of lags (0–14 days)
  - identify lag-at-peak-correlation per station
- 2.3 repeat cross-correlation for wind/precip → salinity, air temp → DO
  - reading suggests DO may be longer lag times
- 2.4 summarize lag structure by regime
  - do cold estuaries respond more slowly than warm ones?
  - other oddities? 
  - stuff we'd report on in paper/poster
- 2.5 identify stations showing trend drift in the STL trend component
  - lag candidates for regime-transition analysis
  - feeds into phase 4, btw...

In [ ]:
# more code!

## Phase 3 — Feature Engineering
Goal: build the lagged, accumulated features your models will actually use

- 3.1 construct rolling-mean atmospheric features at multiple windows
  - 24hr, 72hr, 7-day, 14-day air temp averages
- 3.2 construct rate-of-change features (first diffs) for air temp and wind speed
- 3.3 maybe measure time cyclically?
  - day-of-year as (sin, cos) pair
  - hour-of-day similarly if using 1hr data
  - this just so hours 0 and 23, or days 1 and 365 don't SEEM far apart when they're right next to each other
  - some libraries probably do this already, btw...
- 3.4 construct delta/dff features relative to station's baseline period mean (these will probably do better in a model later)
  - Δ water temp
  - Δ salinity
  - Δ oxygen (dissolved, saturation?)
  - others?

### NOTE -- Phase 3 feeds both 5 and 6.

In [ ]:
# code???

## Phase 4 — Regime Transition Detection
Goal: identify natural validation cases and a forward-projection threshold

- 4.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record
- 4.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory
- 4.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?
- 4.4 designate confirmed transitioning stations as a held-out validation set
  - do not use in model training

### NOTE -- Phase 4 must complete before Phase 5

In [ ]:
# code is the worst ...

## Phase 5 — Model Development: Air → Water Temperature
Goal: predict ΔT_water given atmospheric forcing history

# also stage 1

- 5.1 establish naive baselines
  - persistence forecast and climatological mean by day-of-year
  - record RMSE and MAE
- 5.2 train ridge/lasso regression using lagged atmospheric features
  - interpret coefficients as a sanity check
- 5.3 train gradient boosted model (XGBoost or LightGBM) on same features
  - compare to linear baseline
- 5.4 use walk-forward temporal cross-validation
  - train on years 1–N, test on N+1
  - since reading suggests standard k-fold doesn't work well with this kind of job
- 5.5 evaluate on held-out transitioning stations from phase 4
  - does the model generalize across space?
- 5.6 select best model, document feature importances
  - lag windows that dominate are a reportable finding

In [ ]:
# code code code


## Phase 6 — Model Development: Water Temp → Water Properties
Goal: predict Δsalinity, ΔDO, ΔpH given ΔT_water and atmospheric context

# also stage 2

- 6.1 for each target variable, repeat baseline 
  - linear → gradient boosted sequence from phase 5
- 6.2 use predicted Δ air temp for water as input
  - keeps the pipeline honest and propagates uncertainty
- 6.3 compare response sensitivity by regime
  - does a +2°C warming suppress dissolved oxy more in warm estuaries than cold ones?
  - prolly a yes ... quantifying it is the contribution
- 6.4 document responses as simple lookup relationships
  - e.g., per-degree sensitivities per regime 
  - these are our core scientific deliverables

In [ ]:
# code is okay, I guess ...

## Phase 7 — Nutrient Models
Goal: predict changes in phosphates, nitrates, chlorophyll given water state

# also stage 3 (if time permits)

- 7.1 use the nutrient-inclusive dataset 
  - is okay that coverage is sparser and document that explicitly
- 7.2 same pipeline/process as phase 6, but input features now include water property predictions from earlier
- 7.3 report model skill honestly 
  - nutrient prediction will be noisier; even identifying dominant drivers is a valid result
  - so far, chlorophyll (despite being most obvious to people as blooms) seemed the weakest response?

In [ ]:
# code

## Phase 8 — Forward Projection
Goal: apply response functions to CMIP6 scenarios

- 8.1 obtain CMIP6 projected Δ air temp for your estuary regions under SSP2-4.5 and SSP5-8.5
  - ESGF portal or pre-downscaled products like NASA NEX-GDDP
  - also ote that there's some evidence that CMIP6 has under-estimated climate damage (by overestimating plant growth)
  - and the new estimate thinks it might be about 11% worse than predicted
  - might factor that in (with a *note?)
- 8.2 feed projected Δ air temp into stage 1 model 
  - get projected Δ water temp by decade
- 8.3 Feed projected Δ water temp into stage 2 response functions 
  - get projected Δ salinity, Δ oxygen, Δ pH
- 8.4 Identify projected regime-crossing dates per station under each scenario 
  - *"station X transitions from cool to warm regime between 2045–2060 under SSP5-8.5"* maybe?
- 8.5 propagate and report uncertainty 
  - at minimum, show scenario spread (SSP2 vs SSP5)
  - if time permits, model confidence intervals

In [ ]:
# code, though...